In [10]:
#Model Building & Comparison
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [11]:
# Loading dataset and preparing features for modeling

df = pd.read_csv('../data/raw/Customer_churn.csv')

df = df.drop(columns=['customerID'], errors='ignore')
df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0}).astype(int)

X = df.drop(columns=['Churn']).copy()
X['tenure_monthly_charge_ratio'] = X['tenure'] / (X['MonthlyCharges'] + 1e-6)
X['high_spend_customer'] = (X['MonthlyCharges'] > X['MonthlyCharges'].median()).astype(int)
X['long_term_customer'] = (X['tenure'] >= 12).astype(int)

y = df['Churn']

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train distribution:', y_train.value_counts(normalize=True).to_dict())
print('y_test distribution:', y_test.value_counts(normalize=True).to_dict())

X_train shape: (5634, 5323)
X_test shape: (1409, 5323)
y_train distribution: {0: 0.7346467873624423, 1: 0.2653532126375577}
y_test distribution: {0: 0.7345635202271115, 1: 0.2654364797728886}


In [12]:
#Defining 5 candidate models for comparison using Stratified K-Fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'Extra Trees': Pipeline([
        ('model', ExtraTreesClassifier(n_estimators=200, random_state=42))
    ])
}

models

{'Logistic Regression': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', LogisticRegression(max_iter=5000, random_state=42))]),
 'Decision Tree': Pipeline(steps=[('model', DecisionTreeClassifier(random_state=42))]),
 'Random Forest': Pipeline(steps=[('model',
                  RandomForestClassifier(n_estimators=200, random_state=42))]),
 'Gradient Boosting': Pipeline(steps=[('model', GradientBoostingClassifier(random_state=42))]),
 'Extra Trees': Pipeline(steps=[('model',
                  ExtraTreesClassifier(n_estimators=200, random_state=42))])}

In [14]:
#Running cross-validation for each model and storing the results
cv_results = {}

for name, pipeline in models.items():
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=None
    )
    cv_results[name] = scores
    print(f"{name}: mean F1 = {scores.mean():.4f} | std = {scores.std():.4f}")

Logistic Regression: mean F1 = 0.5723 | std = 0.0102
Decision Tree: mean F1 = 0.5231 | std = 0.0294
Random Forest: mean F1 = 0.5334 | std = 0.0141
Gradient Boosting: mean F1 = 0.5803 | std = 0.0217
Extra Trees: mean F1 = 0.5248 | std = 0.0319


In [15]:
#Training each model on the training set and evaluating on the test set
predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"{name}")
    print(f"  Accuracy : {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall   : {recall:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print('-' * 40)

Logistic Regression
  Accuracy : 0.7807
  Precision: 0.5970
  Recall   : 0.5348
  F1 Score : 0.5642
----------------------------------------
Decision Tree
  Accuracy : 0.7551
  Precision: 0.5413
  Recall   : 0.5080
  F1 Score : 0.5241
----------------------------------------
Random Forest
  Accuracy : 0.7906
  Precision: 0.6426
  Recall   : 0.4759
  F1 Score : 0.5469
----------------------------------------
Gradient Boosting
  Accuracy : 0.8034
  Precision: 0.6690
  Recall   : 0.5134
  F1 Score : 0.5809
----------------------------------------
Extra Trees
  Accuracy : 0.7814
  Precision: 0.6170
  Recall   : 0.4652
  F1 Score : 0.5305
----------------------------------------


In [17]:
#Building a comparison DataFrame to summarize the performance of each model
comparison_rows = []

for name, y_pred in predictions.items():
    comparison_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1_Score': f1_score(y_test, y_pred, zero_division=0),
        'CV_Mean_F1': cv_results[name].mean()
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('F1_Score', ascending=False)
comparison_df

,Model,Accuracy,Precision,Recall,F1_Score,CV_Mean_F1
3,Gradient Boosting,0.803407,0.668990,0.513369,0.580938,0.580283
0,Logistic Regression,0.780696,0.597015,0.534759,0.564175,0.572267
2,Random Forest,0.790632,0.642599,0.475936,0.546851,0.533434
4,Extra Trees,0.781405,0.617021,0.465241,0.530488,0.524785
1,Decision Tree,0.755145,0.541311,0.508021,0.524138,0.523103


In [18]:
for name, y_pred in predictions.items():
    print(f"\n=== {name} Classification Report ===")
    print(classification_report(
        y_test,
        y_pred,
        target_names=['No Churn', 'Churn']
    ))


=== Logistic Regression Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.84      0.87      0.85      1035
       Churn       0.60      0.53      0.56       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.77      0.78      0.78      1409


=== Decision Tree Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.83      0.84      0.84      1035
       Churn       0.54      0.51      0.52       374

    accuracy                           0.76      1409
   macro avg       0.68      0.68      0.68      1409
weighted avg       0.75      0.76      0.75      1409


=== Random Forest Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.86      1035
       Churn       0.64      0.48      0.55       374

    accuracy                           0.